# pymrio parser walkthrough

This walkthrough covers the `mario.parse_from_pymrio(...)` bridge from one in-memory `pymrio.IOSystem` to a `mario.Database`.

## Main entry point

- `mario.parse_from_pymrio(...)`
- upstream object: `pymrio.IOSystem`

## Expected input and assignment patterns

This parser expects one already-loaded `pymrio.IOSystem`, not a filesystem path.

Key arguments:

- `io`: the `pymrio.IOSystem` to convert;
- `value_added`: dictionary or `"all"` selector for the factor side;
- `satellite_account`: dictionary or `"all"` selector for the satellite side;
- `include_meta`: when `True`, copy the upstream metadata trail into the MARIO notes.

Both `value_added` and `satellite_account` can be one explicit dictionary or the shorthand `"all"`. Dictionary values can also be `"all"` or one slicer for part of an extension. With the `"all"` / `"all"` shorthand, MARIO looks for one factor-like extension such as `factor_inputs`, `factor_of_production`, `value_added`, or `primary_inputs`, and assigns the remaining extensions to satellites.

The conversion result is an `IOT`, not a `SUT`.

In [1]:
import mario
import pymrio

## Parse one local EXIOBASE bundle with pymrio

The path below is only an example. Replace it with your local EXIOBASE directory.

In [2]:
io = pymrio.parse_exiobase3(
    path="/path/to/IOT_2013_pxp.zip",
)

## Inspect available pymrio Extensions

This helps you decide whether to pass explicit dictionaries or to rely on the `"all"` shorthand.

In [3]:
[
    name
    for name in dir(io)
    if isinstance(getattr(io, name), pymrio.Extension)
]

['air_emissions',
 'energy',
 'factor_inputs',
 'labour',
 'land',
 'material',
 'nutrients',
 'water']

## Use the full shorthand

When one unique factor-like Extension exists, you can use `value_added="all"` and `satellite_account="all"`. This is practical for EXIOBASE parsed through `pymrio`, where `factor_inputs` is the natural value-added block.

In [4]:
db = mario.parse_from_pymrio(
    io=io,
    value_added="all",
    satellite_account="all",
)

INFO Metadata: initialized.


INFO:mario.api.core_model:Metadata: initialized.


INFO Resolver: resolving v for baseline.


INFO:mario.compute.resolver:Resolver: resolving v for baseline.


INFO Resolver: trying v via formula build_iot_v_from_V_X.


...

INFO:mario.compute.resolver:Resolver: resolving Z for baseline.


INFO Resolver: trying Z via formula build_iot_Z_from_z_X.


INFO:mario.compute.resolver:Resolver: trying Z via formula build_iot_Z_from_z_X.


INFO Resolver: resolved Z via formula build_iot_Z_from_z_X.


INFO:mario.compute.resolver:Resolver: resolved Z via formula build_iot_Z_from_z_X.


In [5]:
db

name = EXIO_IOT_2013_pxp
table = IOT
scenarios = ['baseline']
Factor of production = 9
Satellite account = 725
Consumption category = 7
Region = 49
Sector = 200

## Reference notes and caveats

Upstream documentation: [pymrio documentation](https://pymrio.readthedocs.io/en/latest/index.html).

This is not a file parser. It converts one already-loaded `pymrio.IOSystem`, so any filesystem layout requirements belong to the upstream `pymrio` parser that created `io`.

Practical caveats:

- every `pymrio.Extension` must be classified explicitly, either through one mapping or through the `"all"` shorthand;
- the `"all"` / `"all"` shorthand works only when MARIO can infer one unique factor-like extension;
- this is a conversion bridge, not a generic harmonizer between arbitrary `pymrio` objects and MARIO;
- the resulting MARIO database is currently an `IOT`, not a `SUT`.